# NetForensics AI: An Intelligent Network Fault Detection and Troubleshooting System

## Phase 4: Human Review and Responsible AI


### Project Overview

AI-assisted network troubleshooting systems can provide useful recommendations, but automated results may not always be correct.

The NetForensics AI system uses a human-in-the-loop approach to ensure that important troubleshooting decisions can be reviewed by a network administrator or human expert.

This phase identifies cases that require human review based on the confidence level of the AI-assisted diagnosis.

### Objectives

The objectives of this notebook are:

- Load the AI-assisted diagnosis results.
- Identify cases requiring human review.
- Apply confidence-based review rules.
- Compare automated diagnosis with expected results.
- Create a human review status.
- Maintain transparency in AI-assisted decisions.
- Create a final diagnosis review dataset.

The human review process improves the reliability and responsible use of AI in network troubleshooting.

## 1. Import Required Libraries

The Pandas library is used to load and analyze the AI-assisted diagnosis results.

In [1]:
import pandas as pd

## 2. Load AI-Assisted Diagnosis Results

The results generated during the AI-assisted network fault diagnosis phase are loaded for human review.

The dataset contains the original network fault cases along with similarity scores, confidence levels, probable faults, recommended commands, and suggested fixes.

In [2]:
df = pd.read_csv("ai_assisted_diagnosis_results.csv")

print("AI-assisted diagnosis results loaded successfully.")
print("Number of cases:", len(df))

AI-assisted diagnosis results loaded successfully.
Number of cases: 100


## 3. Display Available Dataset Columns

This step displays the available columns in the dataset before performing the human review analysis.

In [3]:
print(df.columns.tolist())

['Case_ID', 'Category', 'Difficulty', 'Topology', 'Symptom', 'Evidence', 'Expected_Fault', 'OSI_Layer', 'Concept', 'Severity', 'Expected_Next_Command', 'Expected_Fix', 'Verification ', 'Notes', 'AI Prompt', 'Case Text', 'AI_Similar_Case_ID', 'AI_Similarity_Score', 'AI_Confidence', 'AI_Probable_Fault', 'AI_Recommended_Command', 'AI_Suggested_Fix', 'AI_Match']


## 4. Human-in-the-Loop Review Strategy

The NetForensics AI system uses confidence levels to determine whether an automated diagnosis should be reviewed by a human expert.

The review strategy is:

- High Confidence: Automated result can be accepted with minimal review.
- Medium Confidence: Human verification is recommended.
- Low Confidence: Human review is required.

This approach prevents the system from blindly relying on automated predictions.

## 5. Create Human Review Decision Function

A function is created to determine the review status based on the AI confidence level.

The function assigns an appropriate review action to every network troubleshooting case.

In [4]:
def review_decision(confidence):

    if confidence == "High":
        return "Accept - Optional Human Review"

    elif confidence == "Medium":
        return "Human Verification Recommended"

    else:
        return "Human Review Required"

## 6. Apply Human Review Decisions

The human review decision function is applied to all network fault cases.

A new column is created to indicate the required level of human involvement.

In [6]:
df["Human_Review_Status"] = df["AI_Confidence"].apply(review_decision)

print("Human review decisions added successfully.")

Human review decisions added successfully.


## 7. Analyze Human Review Status

This analysis shows how many cases fall into each human review category.

In [ ]:
review_count = df["Human_Review_Status"].value_counts()

print("Human Review Status Distribution:")
print(review_count)

## 8. Identify Cases Requiring Human Review

Cases with medium or low AI confidence require additional verification.

These cases are separated for human review.

In [7]:
review_cases = df[
    df["Human_Review_Status"] != "Accept - Optional Human Review"
]

print("Cases requiring human review:", len(review_cases))

review_cases[[
    "Case_ID",
    "Expected_Fault",
    "AI_Probable_Fault",
    "AI_Confidence",
    "Human_Review_Status"
]].head(10)

Cases requiring human review: 100


,Case_ID,Expected_Fault,AI_Probable_Fault,AI_Confidence,Human_Review_Status
0,C001,PC switch port assigned to wrong VLAN,VLAN 10 missing on Switch1,Medium,Human Verification Recommended
1,C002,VLAN 10 missing on Switch1,PC switch port assigned to wrong VLAN,Medium,Human Verification Recommended
2,C003,VLAN 10 not allowed on trunk,R1 has incorrect static route next hop 10.0.0.3,Medium,Human Verification Recommended
3,C004,DHCP pool configured with wrong network,Missing ip helper-address,Medium,Human Verification Recommended
4,C005,R1 G0/0 administratively shut down,R1 has incorrect static route next hop 10.0.0.3,Medium,Human Verification Recommended
5,C006,PC1 and PC2 configured with the same IP addres...,VLAN 10 not allowed on trunk,Low,Human Review Required
6,C007,R1 has incorrect static route next hop 10.0.0.3,VLAN 10 not allowed on trunk,Medium,Human Verification Recommended
7,C008,PC1 configured with incorrect DNS server 192.1...,Incorrect NTP server configuration,Low,Human Review Required
8,C009,Ethernet cable between Switch1 Fa0/2 and PC2 F...,VLAN 10 not allowed on trunk,Medium,Human Verification Recommended
9,C010,Wrong Ethernet cable type used between PC1 Fa0...,Ethernet cable between Switch1 Fa0/2 and PC2 F...,Low,Human Review Required


## 9. Identify High Confidence Cases

High confidence cases have stronger similarity with existing network fault cases.

These cases can be used as automated troubleshooting recommendations, although a network administrator can still review them if necessary.

In [8]:
high_confidence_cases = df[
    df["AI_Confidence"] == "High"
]

print("High confidence cases:", len(high_confidence_cases))

high_confidence_cases[[
    "Case_ID",
    "Expected_Fault",
    "AI_Probable_Fault",
    "AI_Similarity_Score"
]].head(10)

High confidence cases: 0


,Case_ID,Expected_Fault,AI_Probable_Fault,AI_Similarity_Score


## 10. Create Final Review Recommendation

A final recommendation is created based on both the AI confidence level and the AI diagnosis match result.

This provides an additional layer of validation before accepting an automated diagnosis.

In [9]:
def final_recommendation(match, confidence):

    if match and confidence == "High":
        return "Automated Diagnosis Can Be Accepted"

    elif confidence == "Low":
        return "Manual Investigation Required"

    else:
        return "Human Verification Recommended"

## 11. Apply Final Review Recommendation

The final review recommendation is applied to every network troubleshooting case.

In [10]:
df["Final_Review_Recommendation"] = df.apply(
    lambda row: final_recommendation(
        row["AI_Match"],
        row["AI_Confidence"]
    ),
    axis=1
)

print("Final review recommendations added successfully.")

Final review recommendations added successfully.


## 12. Analyze Final Review Recommendations

This analysis shows the number of cases assigned to each final review recommendation category.

In [11]:
final_review_count = df["Final_Review_Recommendation"].value_counts()

print("Final Review Recommendation Distribution:")
print(final_review_count)

Final Review Recommendation Distribution:
Final_Review_Recommendation
Manual Investigation Required     72
Human Verification Recommended    28
Name: count, dtype: int64


## 13. Explainability of AI-Assisted Diagnosis

Explainability is important in AI-assisted network troubleshooting.

The NetForensics AI system provides information about why a diagnosis was suggested by showing:

- The similar historical case used as a reference.
- The similarity score.
- The confidence level.
- The probable fault.
- The recommended troubleshooting command.
- The suggested fix.

This information allows a human reviewer to understand the basis of the automated recommendation.

In [12]:
explainable_results = df[[
    "Case_ID",
    "AI_Similar_Case_ID",
    "AI_Similarity_Score",
    "AI_Confidence",
    "AI_Probable_Fault",
    "AI_Recommended_Command",
    "AI_Suggested_Fix"
]]

explainable_results.head(10)

,Case_ID,AI_Similar_Case_ID,AI_Similarity_Score,AI_Confidence,AI_Probable_Fault,AI_Recommended_Command,AI_Suggested_Fix
0,C001,C002,0.56,Medium,VLAN 10 missing on Switch1,show vlan brief,Recreate VLAN 10 and assign required ports
1,C002,C001,0.56,Medium,PC switch port assigned to wrong VLAN,show vlan brief,Assign Fa0/1 to VLAN 10
2,C003,C007,0.51,Medium,R1 has incorrect static route next hop 10.0.0.3,show ip route,Change next hop to 10.0.0.2
3,C004,C017,0.42,Medium,Missing ip helper-address,show running-config,ip helper-address on the client-facing router ...
4,C005,C007,0.48,Medium,R1 has incorrect static route next hop 10.0.0.3,show ip route,Change next hop to 10.0.0.2
5,C006,C003,0.31,Low,VLAN 10 not allowed on trunk,show interfaces trunk,Allow VLAN 10 on trunk
6,C007,C003,0.51,Medium,VLAN 10 not allowed on trunk,show interfaces trunk,Allow VLAN 10 on trunk
7,C008,C057,0.25,Low,Incorrect NTP server configuration,show ntp associations,Correct NTP server address/configuration
8,C009,C003,0.44,Medium,VLAN 10 not allowed on trunk,show interfaces trunk,Allow VLAN 10 on trunk
9,C010,C009,0.37,Low,Ethernet cable between Switch1 Fa0/2 and PC2 F...,show interfaces status,Reconnect the Ethernet cable between Switch1 F...


## 14. Responsible AI Limitations

The AI-assisted diagnosis system has certain limitations.

The similarity-based approach depends on the quality and variety of the available network fault cases.

Some limitations include:

- The system may not identify completely new network faults.
- Similar wording does not always indicate the same root cause.
- Low similarity cases may produce unreliable recommendations.
- The dataset size may limit the accuracy of the system.
- Automated recommendations should not replace expert network administrators.

Therefore, low-confidence and uncertain cases should be reviewed by a human expert.

## 15. Create a Human Review Priority

A priority level is assigned to cases based on the confidence of the AI-assisted diagnosis.

This helps human reviewers focus first on cases where the automated system is less confident.

In [13]:
def review_priority(confidence):

    if confidence == "Low":
        return "High Priority"

    elif confidence == "Medium":
        return "Medium Priority"

    else:
        return "Low Priority"

## 16. Apply Human Review Priority

The review priority is added to each network fault case.

Low-confidence cases receive higher priority because they require more careful human investigation.

In [14]:
df["Human_Review_Priority"] = df["AI_Confidence"].apply(review_priority)

print("Human review priority added successfully.")

Human review priority added successfully.


## 17. Create Human Review Log

A separate review log is created for cases that require human verification.

The log contains the original expected fault, AI diagnosis, confidence level, and review priority.

This structure can be used by a network administrator to document the final decision.

In [ ]:
human_review_log = df[
    df["AI_Confidence"] != "High"
][[
    "Case_ID",
    "Expected_Fault",
    "AI_Probable_Fault",
    "AI_Similarity_Score",
    "AI_Confidence",
    "Human_Review_Status",
    "Human_Review_Priority"
]]

human_review_log.head(10)

## 18. Add Human Review Fields

Additional columns are created to support manual review of AI-assisted diagnoses.

A human reviewer can use these fields to record the final decision and comments.

In [15]:
df["Human_Reviewer_Decision"] = "Pending"
df["Human_Reviewer_Comments"] = ""

print("Human review fields added successfully.")

Human review fields added successfully.


## 19. Display Cases Ready for Human Review

This section displays cases that can be reviewed by a network administrator.

The reviewer can compare the AI-assisted diagnosis with the available network evidence before making a final decision.

In [16]:
df[[
    "Case_ID",
    "Symptom",
    "Evidence",
    "Expected_Fault",
    "AI_Probable_Fault",
    "AI_Confidence",
    "Human_Review_Status",
    "Human_Review_Priority",
    "Human_Reviewer_Decision"
]].head(10)

,Case_ID,Symptom,Evidence,Expected_Fault,AI_Probable_Fault,AI_Confidence,Human_Review_Status,Human_Review_Priority,Human_Reviewer_Decision
0,C001,PC cannot reach gateway or server,"show vlan brief, show interfaces fa0/1 switchp...",PC switch port assigned to wrong VLAN,VLAN 10 missing on Switch1,Medium,Human Verification Recommended,Medium Priority,Pending
1,C002,PC cannot reach gateway/server,VLAN and switchport outputs + IP/ping results,VLAN 10 missing on Switch1,PC switch port assigned to wrong VLAN,Medium,Human Verification Recommended,Medium Priority,Pending
2,C003,PC1 cannot reach PC2,show interfaces trunk on Switch1/Switch2; show...,VLAN 10 not allowed on trunk,R1 has incorrect static route next hop 10.0.0.3,Medium,Human Verification Recommended,Medium Priority,Pending
3,C004,PC1 does not receive a vali 192.168.10.x address,"ipconfig, show ip dhcp pool, show ip dhcp bind...",DHCP pool configured with wrong network,Missing ip helper-address,Medium,Human Verification Recommended,Medium Priority,Pending
4,C005,PC1 cannot reach default gateway,"show ip interface brief, show interfaces g0/0,...",R1 G0/0 administratively shut down,R1 has incorrect static route next hop 10.0.0.3,Medium,Human Verification Recommended,Medium Priority,Pending
5,C006,Connectivity fails after two hosts are configu...,"ipconfig, arp -a on PC1/PC2, failed gateway pings",PC1 and PC2 configured with the same IP addres...,VLAN 10 not allowed on trunk,Low,Human Review Required,High Priority,Pending
6,C007,PC1 cannot reach PC2 on remote LAN,"show ip route, show running-config, ipconfig, ...",R1 has incorrect static route next hop 10.0.0.3,VLAN 10 not allowed on trunk,Medium,Human Verification Recommended,Medium Priority,Pending
7,C008,IP connectivity works but hostname resolution ...,"ipconfig, successful DNS-server ping, failed n...",PC1 configured with incorrect DNS server 192.1...,Incorrect NTP server configuration,Low,Human Review Required,High Priority,Pending
8,C009,PC1 cannot communicate with PC2; ping shows 10...,show interfaces status on Switch1 shows Fa0/2 ...,Ethernet cable between Switch1 Fa0/2 and PC2 F...,VLAN 10 not allowed on trunk,Medium,Human Verification Recommended,Medium Priority,Pending
9,C010,PC1 cannot communicate with the switch;Etherne...,show interfaces status shows that Fa0/1 is not...,Wrong Ethernet cable type used between PC1 Fa0...,Ethernet cable between Switch1 Fa0/2 and PC2 F...,Low,Human Review Required,High Priority,Pending


## 20. Human Review Summary

This section provides a summary of cases based on their required level of human involvement.

In [17]:
print("----------------------------------------")
print("NETFORENSICS AI HUMAN REVIEW SUMMARY")
print("----------------------------------------")

print("Total Cases:", len(df))

print("\nHuman Review Status:")
print(df["Human_Review_Status"].value_counts())

print("\nReview Priority:")
print(df["Human_Review_Priority"].value_counts())

print("\nFinal Recommendations:")
print(df["Final_Review_Recommendation"].value_counts())



----------------------------------------
NETFORENSICS AI HUMAN REVIEW SUMMARY
----------------------------------------
Total Cases: 100

Human Review Status:
Human_Review_Status
Human Review Required             72
Human Verification Recommended    28
Name: count, dtype: int64

Review Priority:
Human_Review_Priority
High Priority      72
Medium Priority    28
Name: count, dtype: int64

Final Recommendations:
Final_Review_Recommendation
Manual Investigation Required     72
Human Verification Recommended    28
Name: count, dtype: int64


## 21. Save Human Review Results

The dataset containing the AI-assisted diagnosis and human review information is saved as a new CSV file.

This file represents the final reviewed output of the NetForensics AI analysis pipeline.

In [18]:
df.to_csv("human_review_results.csv", index=False)

print("Human review results saved successfully.")

Human review results saved successfully.


# 22. Conclusion

This phase introduced a human-in-the-loop approach for the NetForensics AI system.

AI-assisted network diagnoses are not blindly accepted. Instead, the confidence level and similarity score are used to determine whether human verification is recommended or required.

The following activities were completed:

- Loaded AI-assisted diagnosis results.
- Created confidence-based human review decisions.
- Identified cases requiring human verification.
- Created final review recommendations.
- Improved explainability by displaying similar cases and similarity scores.
- Defined responsible AI limitations.
- Assigned priorities for human review.
- Created a human review log.
- Added fields for human reviewer decisions and comments.
- Saved the final human review dataset.

The human-in-the-loop approach improves transparency and reliability by ensuring that uncertain AI-assisted diagnoses can be verified by a network administrator.

The next phase will focus on visualization and final project analysis using the generated NetForensics AI results.